# Gemini-3-Flash-Preview ChildSnack (Python generator, unraveling data)

In [ ]:
%pip install -U google-genai

In [ ]:
# Cell 1: Setup imports, paths, and Gemini runtime (independent)
from pathlib import Path
import os
import re
import json
import time
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from google import genai

# Project root (run notebook from formalizer-scaling/src/pddl_gen_notebooks/ChildSnack/single_sandwich_line)
PROJECT_ROOT = Path.cwd().parents[3]

DATA_DIR = PROJECT_ROOT / "data"
PROMPT_PATH = PROJECT_ROOT / "prompts" / "ChildSnack" / "only_pddl_instruction_unraveling.txt"
PDDL_DIR = DATA_DIR / "pddl_unraveling_domain" / "ChildSnack" / "single_sandwich_line"
DESC_DIR = DATA_DIR / "descriptions_unraveling_domain" / "ChildSnack" / "single_sandwich_line"

OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl_unraveling" / "ChildSnack" / "single_sandwich_line" / "gemini_3_flash"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path("/Users/user/Desktop/Drexel NLP/Scalability_project/ours/formalizer-scaling/VAL/build/macos64/Release/bin/Validate")

# Load .env and configure Gemini client
load_dotenv(PROJECT_ROOT / ".env")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env")

client = genai.Client(api_key=GOOGLE_API_KEY)

# Gemini 3 Flash model code (preview)
MODEL_NAME = "gemini-3-flash-preview"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("PDDL dir exists:", PDDL_DIR.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

In [ ]:
# Cell 2: Define target problems and load static prompt text
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 81)]  # p01..p50

PROMPT_PREFIX = PROMPT_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Description dir exists:", DESC_DIR.exists())
print("PDDL dir exists:", PDDL_DIR.exists())

In [ ]:
# Cell 3: Load NL descriptions and ground-truth PDDL for one problem ID
def load_problem_artifacts(problem_id: str):
  domain_nl_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_nl_path = DESC_DIR / f"{problem_id}_problem.txt"
  gt_domain_pddl_path = PDDL_DIR / "domain.pddl"
  gt_problem_pddl_path = PDDL_DIR / f"{problem_id}_problem.pddl"

  if not domain_nl_path.exists():
      raise FileNotFoundError(f"Missing domain NL file: {domain_nl_path}")
  if not problem_nl_path.exists():
      raise FileNotFoundError(f"Missing problem NL file: {problem_nl_path}")
  if not gt_domain_pddl_path.exists():
      raise FileNotFoundError(f"Missing ground-truth domain PDDL: {gt_domain_pddl_path}")
  if not gt_problem_pddl_path.exists():
      raise FileNotFoundError(f"Missing ground-truth problem PDDL: {gt_problem_pddl_path}")

  return {
      "nl_domain": domain_nl_path.read_text(),
      "nl_problem": problem_nl_path.read_text(),
      "gt_domain_pddl_path": gt_domain_pddl_path,
      "gt_problem_pddl_path": gt_problem_pddl_path,
      "gt_domain_pddl_text": gt_domain_pddl_path.read_text(),
  }

In [ ]:
# Cell 4: Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
  artifacts = load_problem_artifacts(problem_id)

  return (
      f"{PROMPT_PREFIX}\n\n"
      "You are an expert PDDL engineer and expert Python engineer.\n"
      "Create an executable Python generator that writes a syntactically and semantically correct "
      "problem file using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{artifacts['nl_domain']}\n\n"
      "### Original problem description\n"
      f"{artifacts['nl_problem']}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{artifacts['gt_domain_pddl_text']}\n\n"
      "Output contract:\n"
      "- Generate ONLY an executable Python generator script.\n"
      "- Wrap it in exactly one <generator>...</generator> block.\n"
      "- Inside <generator>...</generator>, output ONLY raw Python source code.\n"
      "- The first non-whitespace character inside <generator> must be a valid Python token (e.g., def/import).\n"
      "- The script must write ONLY the final problem PDDL to 'problem.pddl' in current working directory.\n"
      "- The script MUST derive init/goal facts algorithmically from compact structure, not hardcoded full fact lists.\n"
      "- Required: at least one loop that constructs predicates (e.g., road/at/in/capacity) from computed patterns.\n"
      "- Required: compute object lists from inferred location count, package count, and vehicle count.\n"
      "- Disallowed: manually listing full init_state/goal_state facts as constants.\n"
      "- Do NOT include Markdown fences (no ```python, no ```).\n"
      "- Do NOT include comments or prose outside Python statements.\n"
      "- Do not output any <domain_file> block.\n"
      "- Do not output any <problem_file> block.\n"
      "- Do not include any extra text before or after <generator>...</generator>."
  )

In [ ]:
# Cell 5: Call Gemini for one prompt (minimal thinking)
def call_gemini(prompt: str, temperature: float = 0.4, max_output_tokens: int = 220000):
  response = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config={
          "temperature": temperature,
          "max_output_tokens": max_output_tokens,
          "thinking_config": {
              "thinking_level": "medium"
          },
      },
  )
  return (response.text or "").strip()

In [ ]:
# Cell 6: Parse model output and extract generator block
def extract_generator_code(raw_text: str):
  text = raw_text or ""
  matches = re.findall(r"<generator>(.*?)</generator>", text, flags=re.DOTALL | re.IGNORECASE)
  if not matches:
      return ""

  # Prefer the last block (models may emit multiple candidate blocks)
  code = (matches[-1] or "").strip()

  # Strip optional markdown fences if present
  if code.startswith("```"):
      code = re.sub(r"^```(?:python)?\s*", "", code, flags=re.IGNORECASE)
      code = re.sub(r"\s*```$", "", code)

  return code.strip()

In [ ]:
# Cell 7: Online solver (public endpoint) + raw API JSON persistence
import requests
import re
import json

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()
  res = None

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  # Persist full final API response for debugging/truncation analysis
  (work_dir / "solver_raw_response.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic for zero-cost empty-plan cases
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [ ]:
# Cell 7a: Revision gate (feedback-driven) for solver failures
def classify_solver_failure_for_revision(solver_result: dict) -> str:
  if solver_result.get("plan_found", False):
      return "plan_found"

  text = "\n".join(
      [
          str(solver_result.get("solver_error", "")),
          str(solver_result.get("stdout", "")),
          str(solver_result.get("stderr", "")),
      ]
  ).lower()

  search_started_markers = [
      "match tree built",
      "pddl problem description loaded",
      "starting search with",
      "--[",
      "goals found",
  ]
  search_finished_markers = [
      "plan found with cost",
      "search completed",
      "total time:",
      "nodes generated during search",
      "nodes expanded during search",
      "problem proven unsolvable",
      "no plan found",
  ]

  started = any(m in text for m in search_started_markers)
  finished = any(m in text for m in search_finished_markers)

  if started and not finished:
      return "cutoff_search_incomplete"   # do not revise
  if started and finished:
      return "completed_no_plan"          # do not revise

  return "pre_search_failure"             # revise


def should_run_revision(solver_result: dict) -> bool:
  return classify_solver_failure_for_revision(solver_result) == "pre_search_failure"


def build_revision_prompt(problem_id: str, previous_problem_pddl: str, previous_generator_code: str, solver_feedback: str) -> str:
  base = build_prompt(problem_id)
  return (
      base
      + "\n\nTo aid the revision, the previous generator code, previous generated problem PDDL, and solver feedback are provided below."
      + "\n------ Previous generator start ------\n"
      + "[GENERATOR CODE]\n"
      + (previous_generator_code or "")
      + "\n------ Previous generator end --------\n\n"
      + "------ Previous problem start ------\n"
      + "[PROBLEM FILE]\n"
      + (previous_problem_pddl or "")
      + "\n------ Previous problem end --------\n\n"
      + "Feedback from the planning solver:\n"
      + (solver_feedback or "Solver returned no plan.")
      + "\n\nRevise the Python generator to fix the issues."
      + " Output only the new generator wrapped in <generator>...</generator>."
  )

In [ ]:
# Cell 7b: Solver wrapper with feedback-driven revisions (up to 3) for generator workflow
import sys
import shutil

def _execute_generator_to_problem(generator_code: str, work_dir: Path, timeout_sec: int = 120):
  work_dir = Path(work_dir)
  work_dir.mkdir(parents=True, exist_ok=True)

  generator_path = work_dir / "generator.py"
  generator_path.write_text(generator_code or "")

  proc = subprocess.run(
      ["python", str(generator_path)],
      cwd=str(work_dir),
      capture_output=True,
      text=True,
      timeout=timeout_sec,
  )

  problem_path = work_dir / "problem.pddl"
  if not (problem_path.exists() and problem_path.read_text().strip()):
      # Fallback: if generator wrote a differently named problem file, copy latest candidate to problem.pddl
      candidates = [
          p for p in work_dir.rglob("*.pddl")
          if p.name not in {"domain.pddl", "problem.pddl"}
      ]
      if candidates:
          latest = max(candidates, key=lambda p: p.stat().st_mtime)
          shutil.copyfile(latest, problem_path)

  ok = problem_path.exists() and bool(problem_path.read_text().strip())
  return {
      "ok": ok and (proc.returncode == 0),
      "returncode": proc.returncode,
      "stdout": proc.stdout or "",
      "stderr": proc.stderr or "",
      "generator_path": str(generator_path),
      "problem_path": str(problem_path),
  }


def run_solver_with_revisions(
  problem_id: str,
  domain_pddl_path: Path,
  problem_pddl_path: Path,
  generator_path: Path,
  work_dir: Path,
  timeout_sec: int = 300,
  max_revisions: int = 0,
):
  problem_pddl_path = Path(problem_pddl_path)
  generator_path = Path(generator_path)
  work_dir = Path(work_dir)

  attempt = 0
  revisions_used = 0
  trace = []
  last_result = None

  while True:
      attempt += 1
      result = run_solver(
          domain_pddl_path=domain_pddl_path,
          problem_pddl_path=problem_pddl_path,
          work_dir=work_dir,
          timeout_sec=timeout_sec,
      )

      cls = classify_solver_failure_for_revision(result)
      print(
          f"[{problem_id}] revision_status: attempt={attempt}, class={cls}, plan_found={bool(result.get('plan_found', False))}",
          file=sys.stderr,
      )
      trace.append(
          {
              "attempt": attempt,
              "plan_found": bool(result.get("plan_found", False)),
              "classification": cls,
              "solver_error": str(result.get("solver_error", "")),
          }
      )
      last_result = result

      if result.get("plan_found", False):
          break

      # No revision for cutoff / completed-no-plan (by policy)
      if not should_run_revision(result):
          print(f"[{problem_id}] no revision triggered (class={cls})", file=sys.stderr)
          break

      if revisions_used >= max_revisions:
          print(f"[{problem_id}] max revisions reached ({max_revisions})", file=sys.stderr)
          break

      prev_problem_text = problem_pddl_path.read_text() if problem_pddl_path.exists() else ""
      prev_generator_text = generator_path.read_text() if generator_path.exists() else ""

      solver_feedback = (
          str(result.get("solver_error", "")).strip()
          or str(result.get("stderr", "")).strip()
          or str(result.get("stdout", "")).strip()
          or "Solver returned no plan."
      )

      print(f"[{problem_id}] revision {revisions_used + 1}/{max_revisions} started", file=sys.stderr)
      revision_prompt = build_revision_prompt(problem_id, prev_problem_text, prev_generator_text, solver_feedback)
      revision_raw = call_gemini(revision_prompt)
      revised_generator = extract_generator_code(revision_raw)

      revisions_used += 1
      rev_dir = work_dir / "revisions"
      rev_dir.mkdir(parents=True, exist_ok=True)
      (rev_dir / f"attempt_{revisions_used:02d}_prompt.txt").write_text(revision_prompt)
      (rev_dir / f"attempt_{revisions_used:02d}_raw.txt").write_text(revision_raw or "")

      if not (revised_generator or "").strip():
          fail = dict(result)
          fail["solver_error"] = (
              str(result.get("solver_error", "")).strip()
              + "\n[revision] model returned no <generator>...</generator> block."
          ).strip()
          last_result = fail
          trace[-1]["revision_generated"] = False
          print(
              f"[{problem_id}] revision {revisions_used}/{max_revisions} failed (no <generator>)",
              file=sys.stderr,
          )
          break

      exec_res = _execute_generator_to_problem(revised_generator, work_dir=work_dir, timeout_sec=120)

      (rev_dir / f"attempt_{revisions_used:02d}_generator.py").write_text(revised_generator)
      (rev_dir / f"attempt_{revisions_used:02d}_exec_stdout.txt").write_text(exec_res.get("stdout", ""))
      (rev_dir / f"attempt_{revisions_used:02d}_exec_stderr.txt").write_text(exec_res.get("stderr", ""))

      if not exec_res.get("ok", False):
          fail = dict(result)
          fail["solver_error"] = (
              str(result.get("solver_error", "")).strip()
              + "\n[revision] generator execution failed to produce valid problem.pddl."
          ).strip()
          last_result = fail
          trace[-1]["revision_generated"] = False
          trace[-1]["generator_exec_ok"] = False
          print(
              f"[{problem_id}] revision {revisions_used}/{max_revisions} failed (generator exec)",
              file=sys.stderr,
          )
          break

      # Persist latest generator and keep revised problem for next solver attempt
      generator_path.write_text(revised_generator)
      trace[-1]["revision_generated"] = True
      trace[-1]["generator_exec_ok"] = True
      print(
          f"[{problem_id}] revision {revisions_used}/{max_revisions} generated new generator.py/problem.pddl",
          file=sys.stderr,
      )

  if last_result is None:
      last_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "internal error: no solver attempts executed",
          "stdout": "",
          "stderr": "",
      }

  out = dict(last_result)
  out["attempts_used"] = attempt
  out["revisions_used"] = revisions_used
  out["retried"] = revisions_used > 0
  out["retry_trace"] = trace
  return out

In [ ]:
# Cell 8: Validate plan with VAL (terminal-equivalent capture, no timeout)
import subprocess
import shlex
import re

def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_file_path: Path):
  cmd = f"{shlex.quote(str(VAL_PATH))} {shlex.quote(str(domain_pddl_path))} {shlex.quote(str(problem_pddl_path))} {shlex.quote(str(plan_file_path))}"
  proc = subprocess.run(
      cmd,
      shell=True,
      capture_output=True,
      text=True,
  )

  val_output = (proc.stdout or "") + ("\n" + proc.stderr if proc.stderr else "")

  ok_patterns = [
      r"Plan valid",
      r"Successful plans:",
  ]
  bad_patterns = [
      r"Plan failed to execute",
      r"Bad plan description",
      r"Syntax error",
      r"Error",
      r"Failed",
      r"Unspecified precondition",
      r"goal not satisfied",
  ]

  val_ok = any(re.search(p, val_output, re.IGNORECASE) for p in ok_patterns) and not any(
      re.search(p, val_output, re.IGNORECASE) for p in bad_patterns
  )

  return {
      "val_ok": bool(val_ok),
      "val_output": val_output,
      "returncode": proc.returncode,
  }

In [ ]:
# Cell 9: Save generated artifacts for one problem
def save_problem_outputs(problem_id: str, raw_text: str, generator_code: str, problem_pddl_text: str, domain_pddl_text: str, solver_result: dict, val_result: dict | None):
  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  # Raw model output
  (cand_dir / "raw.txt").write_text(raw_text or "")

  # Generated Python generator code
  (cand_dir / "generator.py").write_text(generator_code or "")

  # Generated problem PDDL (after executing generator)
  (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")

  # Always store fixed domain used for solve/val (for reproducibility)
  (cand_dir / "domain.pddl").write_text(domain_pddl_text or "")

  # Solver logs
  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  if solver_result.get("plan_found"):
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

  # VAL log (only when VAL was run)
  if val_result is not None:
      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  if "attempts_used" in solver_result:
      (cand_dir / "solver_attempts.txt").write_text(str(solver_result.get("attempts_used", 1)))
  if "retry_trace" in solver_result:
      (cand_dir / "solver_retry_trace.json").write_text(json.dumps(solver_result.get("retry_trace", []), indent=2))
  if "revisions_used" in solver_result:
      (cand_dir / "solver_revisions.txt").write_text(str(solver_result.get("revisions_used", 0)))

  return cand_dir

In [ ]:
# Cell 10: Run one problem through full pipeline
def process_one_problem(problem_id: str):
  artifacts = load_problem_artifacts(problem_id)
  prompt = build_prompt(problem_id)
  raw_text = call_gemini(prompt)

  generator_code = extract_generator_code(raw_text)

  # If model fails to return a tagged generator, record and stop early
  if not generator_code.strip():
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "No <generator>...</generator> found in model output.",
          "stdout": "",
          "stderr": "",
      }
      save_problem_outputs(problem_id, raw_text, "", "", artifacts["gt_domain_pddl_text"], solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  # Write generator first, then execute it to produce problem.pddl
  problem_dir = OUTPUT_ROOT / problem_id / "cand_00"
  problem_dir.mkdir(parents=True, exist_ok=True)

  exec_result = _execute_generator_to_problem(generator_code, work_dir=problem_dir, timeout_sec=120)

  (problem_dir / "generator_exec_stdout.txt").write_text(exec_result.get("stdout", "") or "")
  (problem_dir / "generator_exec_stderr.txt").write_text(exec_result.get("stderr", "") or "")

  if not exec_result.get("ok", False):
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "Generator execution failed to produce valid problem.pddl.",
          "stdout": exec_result.get("stdout", "") or "",
          "stderr": exec_result.get("stderr", "") or "",
      }
      final_problem_pddl_text = (problem_dir / "problem.pddl").read_text() if (problem_dir / "problem.pddl").exists() else ""
      save_problem_outputs(problem_id, raw_text, generator_code, final_problem_pddl_text, artifacts["gt_domain_pddl_text"], solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  gen_problem_path = problem_dir / "problem.pddl"
  gt_domain_pddl_path = artifacts["gt_domain_pddl_path"]
  gt_problem_pddl_path = artifacts["gt_problem_pddl_path"]

  solver_result = run_solver_with_revisions(
      problem_id=problem_id,
      domain_pddl_path=gt_domain_pddl_path,
      problem_pddl_path=gen_problem_path,
      generator_path=problem_dir / "generator.py",
      work_dir=problem_dir,
      timeout_sec=300,
      max_revisions=3,
  )

  val_result = None
  val_ok = False
  if solver_result["plan_found"]:
      plan_path = problem_dir / "plan"
      if not plan_path.exists():
          # fallback safety if solver wrote plan text but not file
          plan_path = problem_dir / "plan.txt"
          plan_path.write_text(solver_result["plan_text"])
      val_result = run_val(gt_domain_pddl_path, gt_problem_pddl_path, plan_path)
      val_ok = bool(val_result["val_ok"])

  # Persist final artifacts (keep latest revised generator/problem if revisions occurred)
  final_problem_pddl_text = gen_problem_path.read_text() if gen_problem_path.exists() else ""
  final_generator_code = (problem_dir / "generator.py").read_text() if (problem_dir / "generator.py").exists() else generator_code
  save_problem_outputs(problem_id, raw_text, final_generator_code, final_problem_pddl_text, artifacts["gt_domain_pddl_text"], solver_result, val_result)

  return {
      "problem": problem_id,
      "parse_ok": True,
      "plan_found": bool(solver_result["plan_found"]),
      "val_ok": val_ok,
      "solver_error": solver_result.get("solver_error", ""),
      "attempts_used": int(solver_result.get("attempts_used", 1)),
  }


In [ ]:
# Cell 11: Generate only (no solver/VAL)
smoke_problem = "p01"
artifacts = load_problem_artifacts(smoke_problem)

prompt = build_prompt(smoke_problem)
raw_text = call_gemini(prompt)
generator_code = extract_generator_code(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

exec_result = {"ok": False, "stdout": "", "stderr": ""}
problem_pddl_text = ""

if generator_code.strip():
  exec_result = _execute_generator_to_problem(generator_code, work_dir=cand_dir, timeout_sec=120)
  if (cand_dir / "problem.pddl").exists():
      problem_pddl_text = (cand_dir / "problem.pddl").read_text()

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "generator.py").write_text(generator_code or "")
(cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
(cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
(cand_dir / "generator_exec_stdout.txt").write_text(exec_result.get("stdout", "") or "")
(cand_dir / "generator_exec_stderr.txt").write_text(exec_result.get("stderr", "") or "")

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_generator_chars:", len(generator_code or ""))
print("generated_problem_pddl_chars:", len(problem_pddl_text or ""))
print("generator_exec_ok:", bool(exec_result.get("ok", False)))
print("saved:", cand_dir)

In [ ]:
# Cell 13: Batch generation-only (with tqdm), default p01..p10
from tqdm.auto import tqdm

BATCH_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(61, 81)]  # change this list for other batches

gen_results = []
for pid in tqdm(BATCH_PROBLEMS, desc="LLM generation", unit="problem"):
  try:
      artifacts = load_problem_artifacts(pid)
      prompt = build_prompt(pid)
      raw_text = call_gemini(prompt)
      generator_code = extract_generator_code(raw_text)

      cand_dir = OUTPUT_ROOT / pid / "cand_00"
      cand_dir.mkdir(parents=True, exist_ok=True)

      exec_result = {"ok": False, "stdout": "", "stderr": ""}
      problem_pddl_text = ""

      if generator_code.strip():
          exec_result = _execute_generator_to_problem(generator_code, work_dir=cand_dir, timeout_sec=120)
          if (cand_dir / "problem.pddl").exists():
              problem_pddl_text = (cand_dir / "problem.pddl").read_text()

      (cand_dir / "raw.txt").write_text(raw_text or "")
      (cand_dir / "generator.py").write_text(generator_code or "")
      (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
      (cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
      (cand_dir / "generator_exec_stdout.txt").write_text(exec_result.get("stdout", "") or "")
      (cand_dir / "generator_exec_stderr.txt").write_text(exec_result.get("stderr", "") or "")

      ok = bool(generator_code.strip()) and bool(exec_result.get("ok", False)) and bool(problem_pddl_text.strip())
      gen_results.append({"problem": pid, "generation_ok": ok, "error": ""})

      if ok:
          print(f"{pid}: generation ok")
      else:
          print(f"{pid}: generation failed (generator missing or execution failed)")
  except Exception as e:
      gen_results.append({"problem": pid, "generation_ok": False, "error": str(e)})
      print(f"{pid}: generation error -> {e}")

print("\nGeneration batch done.")
print("total:", len(gen_results))
print("ok:", sum(1 for r in gen_results if r["generation_ok"]))
print("failed:", sum(1 for r in gen_results if not r["generation_ok"]))

# Parser for Problem Descriptions & Difference Checking

In [ ]:
# Cell 17: Diff section setup (independent, PDDL-vs-PDDL for unraveling)
from pathlib import Path
import re
import json

PROJECT_ROOT = Path.cwd().parents[3]  # run notebook from formalizer-scaling/src/pddl_gen_notebooks/ChildSnack/single_sandwich_line
PDDL_DIR = PROJECT_ROOT / "data" / "pddl_unraveling_domain" / "ChildSnack" / "single_sandwich_line"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl_unraveling" / "ChildSnack" / "single_sandwich_line" / "gemini_3_flash"

# Single-problem handle
DIFF_PROBLEM = "p55"  # edit as needed

# Input files
GT_PDDL_PATH = PDDL_DIR / f"{DIFF_PROBLEM}_problem.pddl"
LLM_PDDL_PATH = OUTPUT_ROOT / DIFF_PROBLEM / "cand_00" / "problem.pddl"

# Output files
CAND_DIR = OUTPUT_ROOT / DIFF_PROBLEM / "cand_00"
CAND_DIR.mkdir(parents=True, exist_ok=True)

DIFF_JSON_OUT = CAND_DIR / "gt_vs_llm_diff.json"
DIFF_TXT_OUT = CAND_DIR / "gt_vs_llm_diff.txt"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Problem:", DIFF_PROBLEM)
print("GT PDDL exists:", GT_PDDL_PATH.exists())
print("LLM PDDL exists:", LLM_PDDL_PATH.exists())
print("Output dir:", CAND_DIR)


In [ ]:
# Cell 18: Helper 1 - normalize whitespace, canonicalize symbols, and extract top-level sections
import re


def _norm_space(s: str) -> str:
  return re.sub(r"\s+", " ", s.strip())


def _canon_symbol(sym: str) -> str:
  s = sym.strip().lower()
  m = re.fullmatch(r"(?:sandwich|sandw|sand|s)-?(\d+)", s)
  if m:
      return f"sandw{m.group(1)}"
  return s


def _extract_section_body(pddl_text: str, section_name: str) -> str:
  marker = f"(:{section_name}"
  start = pddl_text.find(marker)
  if start == -1:
      raise ValueError(f"PDDL missing :{section_name} section.")

  i = start + len(marker)
  depth = 1
  body_chars = []

  while i < len(pddl_text) and depth > 0:
      ch = pddl_text[i]
      if ch == "(":
          depth += 1
      elif ch == ")":
          depth -= 1
          if depth == 0:
              break
      body_chars.append(ch)
      i += 1

  if depth != 0:
      raise ValueError(f"Unbalanced parentheses while reading :{section_name} section.")

  return "".join(body_chars)


In [ ]:
# Cell 19: Helper 2 - extract ChildSnack objects/init/goal from a PDDL problem file
_PREDICATE_RE = re.compile(
  r"\((at_kitchen_bread|at_kitchen_content|no_gluten_bread|no_gluten_content|allergic_gluten|not_allergic_gluten|served|notexist)\s+([^()]+?)\)"
)
_AT_RE = re.compile(r"\(at\s+([^()\s]+)\s+([^()\s]+)\)")
_WAITING_RE = re.compile(r"\(waiting\s+([^()\s]+)\s+([^()\s]+)\)")


def extract_childsnack_problem_structure(pddl_text: str):
  objects_text = _extract_section_body(pddl_text, "objects")
  init_text = _extract_section_body(pddl_text, "init")
  goal_text = _extract_section_body(pddl_text, "goal")

  object_decls = []
  for line in objects_text.splitlines():
      clean = _norm_space(line)
      if not clean or " - " not in clean:
          continue
      names_part, obj_type = clean.rsplit(" - ", 1)
      for name in names_part.split():
          object_decls.append((_canon_symbol(name), obj_type.strip().lower()))

  init_facts = []
  for m in _PREDICATE_RE.finditer(init_text):
      pred = m.group(1)
      args = [_canon_symbol(x) for x in _norm_space(m.group(2)).split()]
      init_facts.append(tuple([pred] + args))
  for m in _AT_RE.finditer(init_text):
      init_facts.append(("at", _canon_symbol(m.group(1).strip()), _canon_symbol(m.group(2).strip())))
  for m in _WAITING_RE.finditer(init_text):
      init_facts.append(("waiting", _canon_symbol(m.group(1).strip()), _canon_symbol(m.group(2).strip())))

  goal_facts = []
  for m in _PREDICATE_RE.finditer(goal_text):
      pred = m.group(1)
      args = [_canon_symbol(x) for x in _norm_space(m.group(2)).split()]
      goal_facts.append(tuple([pred] + args))

  return {
      "objects": object_decls,
      "init_facts": init_facts,
      "goal_facts": goal_facts,
  }


In [ ]:
# Cell 20: Helper 3 - ChildSnack, order-insensitive structural diff (GT vs LLM)
def diff_fact_sections(gt_pddl_text: str, llm_pddl_text: str):
  gt = extract_childsnack_problem_structure(gt_pddl_text)
  llm = extract_childsnack_problem_structure(llm_pddl_text)

  gt_objects = set(gt["objects"])
  llm_objects = set(llm["objects"])
  gt_init = set(gt["init_facts"])
  llm_init = set(llm["init_facts"])
  gt_goal = set(gt["goal_facts"])
  llm_goal = set(llm["goal_facts"])

  return {
      "objects_missing_in_llm": sorted(gt_objects - llm_objects),
      "objects_extra_in_llm": sorted(llm_objects - gt_objects),
      "init_missing_in_llm": sorted(gt_init - llm_init),
      "init_extra_in_llm": sorted(llm_init - gt_init),
      "goal_missing_in_llm": sorted(gt_goal - llm_goal),
      "goal_extra_in_llm": sorted(llm_goal - gt_goal),
      "objects_match_count": len(gt_objects & llm_objects),
      "init_match_count": len(gt_init & llm_init),
      "goal_match_count": len(gt_goal & llm_goal),
      "objects_total_gt": len(gt_objects),
      "objects_total_llm": len(llm_objects),
      "init_total_gt": len(gt_init),
      "init_total_llm": len(llm_init),
      "goal_total_gt": len(gt_goal),
      "goal_total_llm": len(llm_goal),
      "perfect_match": (
          gt_objects == llm_objects and
          gt_init == llm_init and
          gt_goal == llm_goal
      ),
  }


In [ ]:
# Cell 21: Helper 4 - concise mismatch-only diff report text
def build_diff_report_text(problem_id: str, diff_result: dict):
  lines = []
  lines.append(f"Problem: {problem_id}")
  lines.append(f"Perfect match: {diff_result['perfect_match']}")
  lines.append("")

  lines.append(f"Objects missing in LLM ({len(diff_result['objects_missing_in_llm'])}):")
  for obj in diff_result["objects_missing_in_llm"]:
      lines.append(f"- {obj}")

  lines.append("")
  lines.append(f"Objects extra in LLM ({len(diff_result['objects_extra_in_llm'])}):")
  for obj in diff_result["objects_extra_in_llm"]:
      lines.append(f"- {obj}")

  lines.append("")
  lines.append(f"Init missing in LLM ({len(diff_result['init_missing_in_llm'])}):")
  for fact in diff_result["init_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Init extra in LLM ({len(diff_result['init_extra_in_llm'])}):")
  for fact in diff_result["init_extra_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal missing in LLM ({len(diff_result['goal_missing_in_llm'])}):")
  for fact in diff_result["goal_missing_in_llm"]:
      lines.append(f"- {fact}")

  lines.append("")
  lines.append(f"Goal extra in LLM ({len(diff_result['goal_extra_in_llm'])}):")
  for fact in diff_result["goal_extra_in_llm"]:
      lines.append(f"- {fact}")

  return "\n".join(lines) + "\n"


In [ ]:
# Cell 22: Diff run (single problem handle, persists outputs to pXXX/cand_00)
if not GT_PDDL_PATH.exists():
  raise FileNotFoundError(f"Missing ground-truth PDDL: {GT_PDDL_PATH}")
if not LLM_PDDL_PATH.exists():
  raise FileNotFoundError(f"Missing LLM-generated PDDL: {LLM_PDDL_PATH}")

gt_pddl_text = GT_PDDL_PATH.read_text()
llm_pddl_text = LLM_PDDL_PATH.read_text()

diff_result = diff_fact_sections(gt_pddl_text, llm_pddl_text)
diff_text = build_diff_report_text(DIFF_PROBLEM, diff_result)

DIFF_JSON_OUT.write_text(json.dumps(diff_result, indent=2))
DIFF_TXT_OUT.write_text(diff_text)

print("Saved:")
print("-", DIFF_JSON_OUT)
print("-", DIFF_TXT_OUT)
print("")
print("Mismatch summary:")
print(f"- objects missing: {len(diff_result['objects_missing_in_llm'])}")
print(f"- objects extra:   {len(diff_result['objects_extra_in_llm'])}")
print(f"- init missing:    {len(diff_result['init_missing_in_llm'])}")
print(f"- init extra:      {len(diff_result['init_extra_in_llm'])}")
print(f"- goal missing:    {len(diff_result['goal_missing_in_llm'])}")
print(f"- goal extra:      {len(diff_result['goal_extra_in_llm'])}")
print(f"- perfect_match: {diff_result['perfect_match']}")